# Phase 0.5 - Domain-Adaptive Pre-Training (DAPT)

This notebook specialises the multilingual base model to fan discussion. The full cleaned corpus (feature-eligible subreddits plus the control community) is assembled into a training set, the tokenizer is expanded with the curated fandom tokens from Notebook 03 and the new embeddings mean-initialised, and the model is further pre-trained with a masked-language-model objective. The section also documents the GPU-throughput problem caused by a mismatched CUDA build and its resolution.

**Input:** cleaned corpus and candidate tokens. **Output:** the domain-adapted model at `models/dapt_final/`, with held-out perplexity reduced from 9.44 to 5.28.

In [1]:
import os, glob
import pandas as pd
import torch

os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")

CLEAN = "data/interim/clean"
CLEAN_DAPT = "data/interim/clean_dapt"
PROC = "data/processed"
MODELS = "models"
os.makedirs(MODELS, exist_ok=True)

assert torch.cuda.is_available(), "no GPU visible"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

GPU: NVIDIA GeForce RTX 4070 SUPER
VRAM: 12.9 GB


## 1. Assembling the training corpus

Domain adaptation trains on the full corpus: the feature-eligible subreddits together with the r/popculture control community.

In [3]:
# DAPT trains on the FULL corpus: feature-eligible subs + popculture control.
FILES = sorted(glob.glob(f"{CLEAN}/*.parquet") + glob.glob(f"{CLEAN_DAPT}/*.parquet"))
print(len(FILES), "files")

texts = []
for p in FILES:
    d = pd.read_parquet(p, columns=["text"])
    t = d.text.dropna()
    t = t[t.str.split().str.len() >= 10]   # MLM wants sentences, not fragments
    texts.extend(t.tolist())
    del d

print("training texts:", len(texts))
# quick length sanity
import numpy as np
wl = np.array([len(x.split()) for x in texts[:50000]])
print("word-length: median", int(np.median(wl)), "p95", int(np.percentile(wl, 95)))

38 files
training texts: 2407249
word-length: median 30 p95 171


## 2. Expanding the tokenizer

The manually curated fandom tokens are added to the tokenizer and their embeddings mean-initialised from the existing vocabulary, so the new terms start from a sensible representation.

In [4]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
before = len(tok)

new_tokens = pd.read_csv(f"{PROC}/new_tokens.csv").token.astype(str).tolist()
n_added = tok.add_tokens(new_tokens)

print(f"vocab {before} -> {len(tok)}  (+{n_added} added)")
tok.save_pretrained(f"{MODELS}/tok_expanded")

vocab 250002 -> 250316  (+314 added)


('models/tok_expanded/tokenizer_config.json',
 'models/tok_expanded/tokenizer.json')

In [5]:
from transformers import AutoModelForMaskedLM

base_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")   # original, for fragmenting
model = AutoModelForMaskedLM.from_pretrained("xlm-roberta-base")

old_emb = model.get_input_embeddings().weight.data.clone()   # snapshot before resize
model.resize_token_embeddings(len(tok))
emb = model.get_input_embeddings().weight.data

# For each new token, set its embedding = mean of the sub-pieces XLM-R used to split it.
init = 0
for w in new_tokens:
    new_id = tok.convert_tokens_to_ids(w)
    if new_id < len(old_emb):        # token already existed, skip
        continue
    pieces = base_tok(" " + w, add_special_tokens=False)["input_ids"]
    if pieces:
        emb[new_id] = old_emb[pieces].mean(dim=0)
        init += 1

print(f"mean-init applied to {init} tokens")
model.save_pretrained(f"{MODELS}/xlmr_expanded")
tok.save_pretrained(f"{MODELS}/xlmr_expanded")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


mean-init applied to 314 tokens


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/xlmr_expanded/tokenizer_config.json',
 'models/xlmr_expanded/tokenizer.json')

In [6]:
from transformers import AutoModelForMaskedLM

base_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")   # original, for fragmenting
model = AutoModelForMaskedLM.from_pretrained("xlm-roberta-base")

old_emb = model.get_input_embeddings().weight.data.clone()   # snapshot before resize
model.resize_token_embeddings(len(tok))
emb = model.get_input_embeddings().weight.data

# For each new token, set its embedding = mean of the sub-pieces XLM-R used to split it.
init = 0
for w in new_tokens:
    new_id = tok.convert_tokens_to_ids(w)
    if new_id < len(old_emb):        # token already existed, skip
        continue
    pieces = base_tok(" " + w, add_special_tokens=False)["input_ids"]
    if pieces:
        emb[new_id] = old_emb[pieces].mean(dim=0)
        init += 1

print(f"mean-init applied to {init} tokens")
model.save_pretrained(f"{MODELS}/xlmr_expanded")
tok.save_pretrained(f"{MODELS}/xlmr_expanded")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


mean-init applied to 314 tokens


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/xlmr_expanded/tokenizer_config.json',
 'models/xlmr_expanded/tokenizer.json')

## 3. Sub-sampling the training texts

For tractable training time the corpus is capped at 1.2 million texts, sampled with a fixed seed for reproducibility.

In [12]:
import random
random.seed(0)
if len(texts) > 1_200_000:
    texts = random.sample(texts, 1_200_000)
print("DAPT texts:", len(texts))

DAPT texts: 1200000


In [13]:
from datasets import Dataset

MAX_LEN = 192   # covers p95 from Cell 2; raise only if p95 was much higher

ds = Dataset.from_dict({"text": texts}).train_test_split(test_size=0.02, seed=0)

def tok_fn(b):
    return tok(b["text"], truncation=True, max_length=MAX_LEN)

ds = ds.map(tok_fn, batched=True, remove_columns=["text"], num_proc=4)
print(ds)

Map (num_proc=4):   0%|          | 0/1176000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/24000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1176000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 24000
    })
})


## 4. Trial run and throughput check

A short trial run confirms the training loop is functioning and surfaces the GPU-utilisation problem addressed below.


In [7]:
import time, torch, subprocess
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

# Make sure MAX_LEN=192 was set in Cell 5 and the dataset re-tokenised before this.

coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=0.15)

time_args = TrainingArguments(
    output_dir=f"{MODELS}/dapt_timing",
    max_steps=50,
    per_device_train_batch_size=12,
    gradient_accumulation_steps=6,
    dataloader_num_workers=0,        # start at 0 (WSL-safe); try 2 if this is slow
    dataloader_pin_memory=True,
    bf16=True,
    fp16=False,
    logging_steps=50,
    report_to="none",
)

# --- warm up: compile CUDA kernels, spin loaders — this run is IGNORED ---
warm = Trainer(model=model, args=time_args,
               train_dataset=ds["train"].select(range(500)),
               data_collator=coll)
warm.train()

# --- timed measurement on a fresh slice ---
torch.cuda.synchronize()
t0 = time.time()
timed = Trainer(model=model, args=time_args,
                train_dataset=ds["train"].select(range(1500)),
                data_collator=coll)
timed.train()
torch.cuda.synchronize()
dt = time.time() - t0

per_step = dt / time_args.max_steps
steps_per_epoch = 2_360_000 // (12 * 6)     # effective batch 72
print(f"\n{'='*40}")
print(f"warmed-up speed : {per_step:.2f} s/step")
print(f"steps/epoch     : {steps_per_epoch:,}")
print(f"1-epoch runtime : {per_step * steps_per_epoch / 3600:.1f} hours")
print(f"{'='*40}")
print(subprocess.run(["nvidia-smi","--query-gpu=utilization.gpu,memory.used",
                      "--format=csv"], capture_output=True, text=True).stdout)

Step,Training Loss
50,13.013082


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
50,13.150604


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


warmed-up speed : 19.15 s/step
steps/epoch     : 32,777
1-epoch runtime : 174.3 hours
utilization.gpu [%], memory.used [MiB]
19 %, 11766 MiB



## 5. Training configuration

In [14]:
import math
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=0.15)

args = TrainingArguments(
    output_dir=f"{MODELS}/dapt",
    num_train_epochs=1,                     # 1 pass — corpus is large enough
    per_device_train_batch_size=24,         # drop to 16 if the OOM check failed
    gradient_accumulation_steps=3,          # effective batch ~72 (use 4 if bs=16)
    dataloader_num_workers=0,               # WSL-safe; workers slowed it earlier
    dataloader_pin_memory=True,
    bf16=True,
    fp16=False,
    learning_rate=5e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    logging_steps=200,
    eval_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    save_total_limit=2,                     # keeps only 2 latest checkpoints (disk)
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=coll,
)

# perplexity BEFORE adaptation — your baseline for the results chapter
pre = trainer.evaluate()
print("perplexity BEFORE:", round(math.exp(pre["eval_loss"]), 2))

# the long run (~14h at bs=24). If it dies, resume with:
#   trainer.train(resume_from_checkpoint=True)
trainer.train()

# perplexity AFTER — evidence DAPT worked
post = trainer.evaluate()
print("perplexity AFTER:", round(math.exp(post["eval_loss"]), 2))

trainer.save_model(f"{MODELS}/dapt_final")
tok.save_pretrained(f"{MODELS}/dapt_final")
print("saved -> models/dapt_final")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training Loss,Validation Loss,Step
No log,2.244667,0


perplexity BEFORE: 9.44


Step,Training Loss,Validation Loss
2000,6.164725,1.934809
4000,5.897185,1.860751
6000,5.733461,1.809600
8000,5.645418,1.770711
10000,5.512129,1.734424
12000,5.392858,1.710664
14000,5.292620,1.694528
16000,5.305087,1.667779
16334,5.282957,1.673877


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step
5.282957,1.663283,16334


perplexity AFTER: 5.28


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved -> models/dapt_final


In [9]:
from transformers import (DataCollatorForLanguageModeling, Trainer,
                          TrainingArguments)

coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=0.15)

trial_args = TrainingArguments(
    output_dir=f"{MODELS}/dapt_trial",
    max_steps=50,                       # tiny — just proving it fits
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    fp16=True,
    logging_steps=10,
    report_to="none",
)

trial = Trainer(model=model, args=trial_args,
                train_dataset=ds["train"].select(range(2000)),
                data_collator=coll)
trial.train()
print("trial OK — batch size fits")

Step,Training Loss
10,9.528457
20,9.453078
30,9.420285
40,8.334378
50,9.001308


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

trial OK — batch size fits


In [10]:
print("model device:", next(model.parameters()).device)
print("cuda available:", torch.cuda.is_available())
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=utilization.gpu,memory.used",
                      "--format=csv"], capture_output=True, text=True).stdout)


model device: cuda:0
cuda available: True
utilization.gpu [%], memory.used [MiB]
39 %, 11778 MiB



due to low gpu utilization here are some tweaks


In [11]:
args = TrainingArguments(
    output_dir=f"{MODELS}/dapt",
    num_train_epochs=3,
    per_device_train_batch_size=12,        # 16 -> 12: buys VRAM headroom
    gradient_accumulation_steps=6,          # 4 -> 6: keeps effective batch ~72
    dataloader_num_workers=4,               # THE FIX: parallel data loading
    dataloader_pin_memory=True,             # faster host->GPU transfer
    bf16=True,                              # replaces fp16 — more stable, same speed on 40-series
    fp16=False,
    learning_rate=5e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    logging_steps=200,
    eval_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    save_total_limit=2,
    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


VERIFICATION


In [9]:
import torch, time

# --- 1. verify the torch reinstall landed ---
print("torch:", torch.__version__)          # want +cu121, NOT +cu130
print("cuda build:", torch.version.cuda)     # want 12.1
print("cuda available:", torch.cuda.is_available())
import sys; print("python:", sys.executable) # must be inside /root/tf-project/venv
print("-" * 40)

# --- 2. put the model on the GPU (kernel restart reset it to CPU) ---
model = model.to("cuda")
print("model device:", next(model.parameters()).device)   # must say cuda:0
print("-" * 40)

# --- 3. raw GPU speed test ---
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
torch.cuda.synchronize(); t0 = time.time()
for _ in range(100):
    y = x @ x
torch.cuda.synchronize()
print(f"matmul 100x: {time.time()-t0:.2f}s   (healthy ≈ 0.3s)")

# --- 4. model forward+backward test ---
from transformers import DataCollatorForLanguageModeling
coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=0.15)
batch = coll([ds["train"][i] for i in range(12)])
batch = {k: v.to("cuda") for k, v in batch.items()}

model.train()
torch.cuda.synchronize(); t0 = time.time()
for _ in range(10):
    out = model(**batch)
    out.loss.backward()
    model.zero_grad()
torch.cuda.synchronize()
dt = time.time() - t0
print(f"10 fwd+bwd (bs=12): {dt:.2f}s   ({dt/10:.2f}s/iter, healthy ≈ 0.4s/iter)")

torch: 2.5.1+cu121
cuda build: 12.1
cuda available: True
python: /root/tf-project/venv/bin/python
----------------------------------------
model device: cuda:0
----------------------------------------
matmul 100x: 0.39s   (healthy ≈ 0.3s)
10 fwd+bwd (bs=12): 30.35s   (3.03s/iter, healthy ≈ 0.4s/iter)


In [10]:
print("batch input shape:", batch["input_ids"].shape)

batch input shape: torch.Size([12, 192])


In [11]:
import torch, time

print("batch shape:", batch["input_ids"].shape)

model.train()
torch.cuda.synchronize(); t0 = time.time()
for _ in range(10):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        out = model(**batch)
    out.loss.backward()
    model.zero_grad()
torch.cuda.synchronize()
dt = time.time() - t0
print(f"10 fwd+bwd (bf16): {dt:.2f}s  ({dt/10:.2f}s/iter)")


batch shape: torch.Size([12, 192])
10 fwd+bwd (bf16): 5.34s  (0.53s/iter)


In [6]:
import glob, os
os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")  # go to project root
print("cwd:", os.getcwd())

cks = sorted(glob.glob("models/dapt/checkpoint-*"),
             key=lambda p: int(p.split("-")[-1]))
print("checkpoints:", cks)
print("dapt_final exists:", os.path.exists("models/dapt_final"))
print("dapt_final contents:", os.listdir("models/dapt_final") if os.path.exists("models/dapt_final") else "empty")

cwd: /root/tf-project/venv/Projects/BehaviouralAnalysis
checkpoints: ['models/dapt/checkpoint-16000', 'models/dapt/checkpoint-16334']
dapt_final exists: True
dapt_final contents: ['tokenizer.json', 'training_args.bin', 'config.json', 'tokenizer_config.json', 'model.safetensors']


In [8]:
from transformers import AutoTokenizer, pipeline

tok = AutoTokenizer.from_pretrained("models/dapt_final")
fill_new  = pipeline("fill-mask", model="models/dapt_final", tokenizer=tok)
fill_base = pipeline("fill-mask", model="xlm-roberta-base")

probes = [
    "My <mask> is my ultimate favourite member.",
    "I spent all my money on <mask> this comeback.",
    "The antis started <mask> her online.",
    "She is my <mask>, I would do anything for her.",
]
for s in probes:
    print(s)
    print("  DAPT:", [r["token_str"].strip() for r in fill_new(s)[:5]])
    print("  base:", [r["token_str"].strip() for r in fill_base(s)[:5]])
    print()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


My <mask> is my ultimate favourite member.
  DAPT: ['friend', 'favourite', 'sister', 'husband', 'boss']
  base: ['friend', 'boyfriend', 'husband', 'brother', 'daughter']

I spent all my money on <mask> this comeback.
  DAPT: ['them', 'BTS', 'this', 'it', 'twice']
  base: ['making', 'watching', 'getting', 'creating', 'doing']

The antis started <mask> her online.
  DAPT: ['following', 'calling', 'fighting', 'using', 'selling']
  base: ['dating', 'selling', 'helping', 'following', 'seeing']

She is my <mask>, I would do anything for her.
  DAPT: ['heart', 'friend', 'sister', 'favorite', 'fan']
  base: ['friend', 'best', 'daughter', 'mother', 'heart']

